# exp100 pf_z unified velocity observation prior train

Stage 1 train pseudo-tail ablation for weak XY, prefix-slope, and GR-calibrated priors inside `pf_z` particle weights.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from pf_z_unified_velocity_observation_prior import run_audit

config = load_config()
paths = ExperimentPaths()

print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('status:', get_nested(config, 'experiment.status'))
print('parent:', get_nested(config, 'lineage.parent'))
print('train_dir:', paths.train_data_dir)
print('artifacts_dir:', paths.artifacts_dir)
print('variants:', [item['name'] for item in get_nested(config, 'model.variants')])
print('pf_z:', json.dumps(get_nested(config, 'model.pf_z'), indent=2, sort_keys=True))

## 2. Input preview

In [ ]:
horizontal_files = sorted(paths.train_data_dir.glob('*__horizontal_well.csv'))
typewell_files = sorted(paths.train_data_dir.glob('*__typewell.csv'))
print('horizontal files:', len(horizontal_files))
print('typewell files:', len(typewell_files))
assert horizontal_files, 'No train horizontal well files found'
sample_horizontal = pd.read_csv(horizontal_files[0], nrows=8)
sample_typewell = pd.read_csv(paths.train_data_dir / horizontal_files[0].name.replace('__horizontal_well.csv', '__typewell.csv'), nrows=8)
display(sample_horizontal)
display(sample_typewell)
print('sample eval rows:', int(sample_horizontal['TVT_input'].isna().sum()))

## 3. Run PF z-prior ablation

In [ ]:
summary = run_audit(config)
print(json.dumps(summary, indent=2, sort_keys=True)[:8000])

## 4. Metrics and artifacts

In [ ]:
artifact_dir = paths.artifacts_dir
variant_metrics_path = artifact_dir / 'exp100_pf_z_unified_velocity_observation_prior_variant_metrics.csv'
bucket_metrics_path = artifact_dir / 'exp100_pf_z_unified_velocity_observation_prior_bucket_metrics.csv'
by_well_path = artifact_dir / 'exp100_pf_z_unified_velocity_observation_prior_by_well.csv'
summary_path = artifact_dir / 'exp100_pf_z_unified_velocity_observation_prior_summary.json'

variant_metrics = pd.read_csv(variant_metrics_path)
bucket_metrics = pd.read_csv(bucket_metrics_path)
by_well = pd.read_csv(by_well_path)

display(variant_metrics)
display(bucket_metrics.head(20))
display(by_well.sort_values('rmse', ascending=False).head(20))
print('summary:', summary_path)
print('artifact files:')
for path in sorted(artifact_dir.glob('exp100_pf_z_unified_velocity_observation_prior*')):
    print(path.name, path.stat().st_size)